# Project2 实验报告
#### CS30064.01 Neural Network and Deep Learning
##### 洪家权 23307110248

## 1. 在 CIFAR-10 上训练网络

### 1.1.数据集使用说明

本任务采用 CIFAR-10 彩色图像分类数据集，共 60,000 张 32×32 图像、10 个类别。其中，有 50,000 张为官方训练集，在本实验任务中作为训练集与验证集的数据来源，有 10,000 张为官方测试集，独立保留，不参与训练集验证集划分，仅用于在每一次模型训练实验结束后得到的最优权重上作评估。

在实际训练中，将官方 50,000 张训练图像按 **9 : 1** 随机划分为45,000 张**训练集**与5,000 张**验证集**，划分时使用固定随机种子 **2020**，保证各次实验、各组消融使用同一批样本索引。

对数据集分别作以下数据预处理：

- **训练集**：在 32×32 图像上做随机裁剪（四周各补 4 像素后再裁回 32×32）、随机水平翻转，再转为张量，并对 RGB 三通道做归一化（均值 0.5、标准差 0.5）。  
- **验证集与测试集**：仅做张量转换与相同归一化，**不使用**随机裁剪与翻转，以便稳定评估。

下面给出训练集、验证集与测试集的使用方式：**训练集**在每一轮训练中用于前向、反向与参数更新；**验证集**不参与梯度更新，在每个 epoch 训练结束后，在其上计算验证损失与准确率，并**保存验证准确率最高**的模型权重，供后续测试与对比；**测试集**全程不参与训练与超参选择，全部训练结束后，仅用上述验证集最优权重在测试集上**评估一次**，得到报告中的最终测试误差/准确率。

### 1.2. 网络结构

本任务采用面向 CIFAR-10 设计的 **残差卷积网络（ResNet 风格）**，记为 **CifarResNet**。下文以 **baseline** 配置（通道数 64–128–256–512、每阶段 2 个残差块、ReLU 激活、不使用 Dropout）为主说明数据流与各层作用；消融实验中调整了通道宽度、激活函数、Dropout 等，但主干拓扑结构维持不变。

#### 1.2.1 整体数据流与特征图尺寸

输入一张 32×32×3 的图像后，特征图空间尺寸与通道数变化如下：

| 阶段 | 输出空间尺寸 | 输出通道数 | 说明 |
|------|--------------|------------|------|
| 输入 | 32×32 | 3 | RGB 图像 |
| 入口层 | 32×32 | 64 | 不改变分辨率，仅提升通道数 |
| 残差阶段 1 | 32×32 | 64 | 两个残差块，不下采样 |
| 残差阶段 2 | 16×16 | 128 | 首个块将边长减半 |
| 残差阶段 3 | 8×8 | 256 | 同上 |
| 残差阶段 4 | 4×4 | 512 | 同上 |
| 全局平均池化 | 1×1 | 512 | 将 4×4 特征图池化为单个向量 |
| 分类层 | — | 10 | 输出 10 类 logits |

网络未使用传统的固定大小最大池化堆叠下采样，而是主要在残差阶段通过 **步长为 2 的卷积** 降低空间分辨率；最后用 **全局平均池化** 代替大尺寸全连接层前的展平，使分类头参数量较小。

#### 1.2.2 入口层（Stem）

入口层对原始图像做一次 3×3 卷积，将通道数从 3 提升到 64，步长为 1、保持 32×32 分辨率；随后进行批归一化与非线性激活（baseline 为 ReLU）。该层作用类似于 ResNet 的 stem，为后续残差阶段提供初始特征表示。

#### 1.2.3 残差阶段与残差块

网络主体由四个残差阶段串联而成，每阶段堆叠 **2 个** 结构相同的基础残差块。单个残差块含 **主路径** 与 **捷径路径**：主路径上依次经过两个 3×3 卷积（各在卷积后进行 **批归一化** ），第一个卷积的步长可大于 1 以完成下采样或升通道，第二个卷积步长恒为 1，中间插入激活函数；捷径路径将块输入与主路径输出 **逐元素相加**，若二者尺寸或通道不一致，则对输入做 1×1 卷积（含批归一化）对齐后再相加，否则直接恒等传递；相加后再经激活得到块输出。

各阶段通过控制 **首个块的步长与输出通道**，实现分辨率与语义层次的递进；同阶段第二个块步长为 1，仅在当前分辨率上加深特征变换。**阶段 1** 在 32×32、64 通道下堆叠两块，不降低分辨率，侧重纹理与边缘等低级特征；**阶段 2** 由首个块将特征图减至 16×16、通道升至 128，第二个块保持该尺寸，引入更丰富局部模式；**阶段 3** 进一步得到 8×8、256 通道中层语义表征；**阶段 4** 输出 4×4、512 通道的高层语义特征，供后续分类使用。另外，baseline 下通道宽度依次为 64→128→256→512，宽度消融时对各阶段通道整体缩放以考察模型容量的影响，具体设置见后文。

#### 1.2.4 分类头

经全局平均池化后，每个样本得到长度为 512 的特征向量。baseline **不使用 Dropout**，直接经一层全连接映射到 10 维输出，对应 CIFAR-10 十个类别，训练时对输出施加交叉熵损失。后续实验中启用 Dropout 时， **仅在全连接之前** 以一定概率随机置零部分特征，用于缓解过拟合。

### 1.3. 基线实验

#### 1.3.1. 实验设置

本小节给出基线实验的完整训练配置。

网络采用 1.2 节所述 CifarResNet 的默认拓扑，数据划分与预处理同 1.1 节；训练小批量为 **512**，固定训练 **200** 个 epoch，**不使用早停**；随机种子 **2020** 用于数据划分与参数初始化；损失函数为 **交叉熵**，不启用标签平滑、Mixup 或 CutMix；优化器采用 **AdamW**，初始学习率 **1×10⁻³**，权重衰减（L2）为 **5×10⁻⁴**（decoupled weight decay，与一阶动量更新解耦）；学习率按 **余弦退火** 从 **1×10⁻³** 起在 200 个 epoch 内单调衰减至接近 0，**每个 epoch 结束后** 更新一次；如前所述，每个 epoch 在验证集上计算损失与准确率，**保存验证准确率最高** 的权重，全部训练结束后，用该权重在官方测试集上 **仅评估一次**，得到基线的最终测试误差/准确率。后续 1.4–1.8 节消融在保持上述流程不变的前提下，分别只改动通道宽度、损失函数、正则化强度、添加正则化方式或激活函数，或优化器及其学习率。

#### 1.3.2. 实验结果

下表为基线实验结果汇总。

| 实验 | 参数量 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|--------|------------------|------------|--------------|
| 基线（AdamW 1e-3） | 11.17M | 0.9318 | 183 | 0.9308 |

由表可知，基线在约第 183 个 epoch 达到最优验证准确率 0.9318，测试准确率 0.9308，二者接近，说明在该配置下过拟合并不严重，可作为后续消融的统一对照。收敛速度上，基线在 epoch 183 达到验证峰值，表明基线实验到训练中后期才饱和。

下图为基线实验验证集与训练集的 loss 与 accuracy 曲线。

![1.3.2 基线实验曲线](pic/1_3_2.png)

由图可知， 训练/验证 loss 同步下降、accuracy 同步上升，且验证曲线与训练曲线贴合，与表中验证 0.9318、测试 0.9308 的接近一致；后期未出现验证指标明显恶化，从曲线侧印证了过拟合不严重。

### 1.4. 实验1：尝试不同数量的神经元/卷积核

#### 1.4.1. 实验设置

在 **1.3.1 基线** 基础上，**仅改变各残差阶段的卷积通道数**，其余所有设置均与基线模型一致。

共进行 **2 组** 对照：

| 实验 | 四阶段通道 | 相对基线 |
|------|------------------------|----------|
| 窄网络 | 32 → 64 → 128 → 256 | 通道数为基线通道的一半，参数量更小 |
| 基线模型 | 64 → 128 → 256 → 512 | 无 |
| 宽网络 | 96 → 192 → 384 → 768 | 通道数为基线通道的 1.5 倍，容量更大 |

实验的目的是，考察在相同训练超参下，**模型容量（通道宽度）** 对验证/测试分类性能与训练稳定性的影响。

#### 1.4.2. 实验结果

下表为网络宽度消融实验结果汇总。

| 实验 | 参数量 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|--------|------------------|------------|--------------|
| 基线 | 11.17M | 0.9318 | 183 | 0.9308 |
| 窄网络 | 2.80M | 0.9134 | 195 | 0.9107 |
| 宽网络 | 25.13M | 0.9378 | 159 | 0.9346 |

由表可知，参数量随通道宽度约为 **2.80M / 11.17M / 25.13M**，且随通道宽度增加，验证/测试准确率整体上升。收敛速度上，同等训练代数预算的情况下，宽网络最早（159）、基线次之（183）、窄网络最晚（195）达到最优性能，表明适度增加模型容量有助于更快达到较优验证点；但窄网络降幅有限，且基线宽度已具备较好性价比。

下图为宽度消融实验验证集与训练集的 loss 与 accuracy 曲线。

![1.4.2 宽度消融曲线](pic/1_4_2.png)

由图可知， 宽网络验证 accuracy 上升最快、最终最高，窄网络全程低于基线，与表中 0.9346 / 0.9308 / 0.9107 的排序一致；宽网络 loss 下降更陡，说明容量增大加快了拟合，但也需结合表中仅小幅领先基线来看，收益存在边际递减。从训练集的损失与准确率曲线来看，窄网络收敛速度显著慢于其他两组，宽网络与基线基本持平，这是因为窄网络容量偏小，相同样本上参数更新对训练损失的压降较慢，而适度增宽后拟合能力增强，训练曲线下降更接近基线水平。这与前文图表解读的结论相呼应，表明模型容量对于整体学习进度以及收敛速度有着显著影响。

### 1.5. 实验2：尝试不同的损失函数

#### 1.5.1. 实验设置

本实验在 **1.3.1 基线** 基础上 **仅替换分类损失函数**，其余设置均与基线一致。

共进行 **2 组** 对照：

| 实验 | 损失函数 | 说明 |
|------|----------|------|
| 基线（对照） | **交叉熵（CE）** | 见 1.3.1，本组不重复训练 |
| Focal Loss | **Focal 损失，聚焦参数 γ = 2** | 在 CE 基础上对难分类样本赋予更大梯度权重，缓解易样本主导 |
| Multi-Margin Loss | **多类间隔损失，margin = 1，范数 p = 2** | 在 logits 上拉大正确类与错误类的间隔（SVM / hinge 风格），见下文式 (3) |

设网络对单个样本输出 **logits** 向量 $\mathbf{z}=(z_1,\ldots,z_K)^\top$（$K=10$），真实类别为 $y$。三种损失在本实验中的定义如下：

**(1) 交叉熵**

交叉熵损失的定义为：

$$
\mathcal{L}_{\mathrm{CE}} = -\log \frac{e^{z_y}}{\sum_{k=1}^{K} e^{z_k}} = -z_y + \log\sum_{k=1}^{K} e^{z_k}.
$$

等价于令 Softmax 概率 $p_k = e^{z_k}/\sum_j e^{z_j}$，最小化 $-\log p_y$，即**极大化正确类概率**。优化目标是让预测分布逼近 **one-hot 标签**，同时隐含的优化目标是，拉近正确 logit、压低错误类 logit。

**(2) Focal Loss（$\gamma=2$）**

定义样本的 CE：

$$\mathrm{CE} = -\log p_y$$

定义**正确类概率**：

$$p_t = p_y = e^{-\mathrm{CE}}$$

则

$$
\mathcal{L}_{\mathrm{Focal}} = (1 - p_t)^{\gamma}\,\mathrm{CE}, \quad \gamma=2.
$$

其中 $(1-p_t)^{\gamma}$ 为调制因子。$p_t\to 1$（易分样本）时因子趋于0，损失与梯度被压低；$p_t$ 小（难分样本）时因子 $\to 1$，该损失函数的行为接近 CE。这一损失函数的主要优化目标仍是提高 $p_y$，但**刻意降低易分类样本对总损失的贡献**，使梯度更多来自难例/边界样本，缓解类别不平衡或大量已学会样本主导更新。

**(3) Multi-Margin Loss（$m=1$，$p=2$）**

对所有错误类 $j\neq y$ 施加间隔约束：

$$
\mathcal{L}_{\mathrm{MM}} = \frac{1}{K-1}\sum_{j\neq y} \Big[\max\big(0,\; m - (z_y - z_j)\big)\Big]^2
$$

该损失函数的意义是，当正确类 logit 比第 $j$ 类至少大 $m$ 时，该项为 0；否则按 **hinge** 形式惩罚，$p=2$ 时为平方 hinge ，类似于 SVM 的多类分割间隔。这一损失函数的优化目标与前两者不同：其**不拟合概率分布**，而是要求样本的正确类 $z_y$ 相对每一错误类 $z_j$ 留出固定间隔 $m$；关心的是正确 logit 与错误 logit 的**相对大小与间隔**，而非 $p_y$ 是否接近 1。

鉴于三种损失函数的 loss 量纲与数值尺度不同 (CE / Focal 以 $-\log p_y$ 为基础,而 Multi-Margin 是 margin hinge 的平方和，最优值可接近 0，收敛平台与 CE 无固定比例关系)，而且 loss 的最优取值与模型选优的对应关系不同(前两者损失数值小通常表示 $p_y$ 大,而后者损失数值小表示间隔约束大多满足，**同一数值不表示同等分类置信度**)因此其 loss 不具备可比性。故在1.5.2的绘图部分，不会给出 loss 曲线，只给出 accuracy 曲线。

本实验的目的是，在固定结构与训练超参下，比较三种损失函数对收敛行为与最终分类性能的差异，注意与 1.6 节中正则化及数据增强类消融相区分。

#### 1.5.2. 实验结果

下表为损失函数消融实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| 基线（CE） | 0.9318 | 183 | 0.9308 |
| Focal Loss | 0.9320 | 150 | 0.9236 |
| Multi-Margin Loss | 0.9280 | 164 | 0.9288 |

由表可知，三种损失在验证集上差距不大（0.9280–0.9320），但 Focal Loss 测试准确率明显低于基线（0.9236 vs. 0.9308），Multi-Margin 亦略低；交叉熵在本设置下最终泛化最稳。Focal 虽更早达到较优验证点（epoch 150），却未能转化为同等测试性能。收敛速度上，Focal Loss 最早（150）、Multi-Margin 次之（164）、CE 最晚（183）达到最优性能；Focal 虽收敛最快，却未带来更优测试精度。

下图为损失函数消融实验验证集与训练集的 accuracy 曲线。

![1.5.2 损失函数消融曲线](pic/1_5_2.png)

由图可知， 中后期 CE 的验证 accuracy 整体最高且最稳，Focal 虽在 epoch 150 附近一度与 CE 接近，但后期验证曲线回落，与表中测试仅 0.9236 相呼应；Multi-Margin 略低于 CE，曲线波动更大，说明间隔损失在本任务上未带来优于 CE 的泛化。从训练集准确率曲线来看，CE 收敛快于 Focal Loss，Focal Loss 收敛快于 Multi-Margin Loss，这是因为 CE 直接优化类概率，初期梯度信号更直接，Focal 对易样本降权使中后期有效更新偏慢，Multi-Margin 以间隔约束为主，训练早期 logits 尚未拉开时损失下降相对滞后。

### 1.6. 实验3：尝试不同的正则化

#### 1.6.1. 实验设置

在 **1.3.1 基线** 基础上，**仅改变损失目标或与之配套的正则化手段**，其余设置均保持与基线模型一致。

下表为正则化消融实验的设置说明。

| 实验 | 相对基线的改动 | 说明 |
|------|----------------|------|
| 无权重衰减 | 权重衰减 **0** | 仅保留交叉熵，考察 L2 正则关闭的影响 |
| 基线模型(对照) | 权重衰减 **5×10⁻⁴** | 对照组 |
| 强权重衰减 | 权重衰减 **1×10⁻³** | 加强 L2，与基线 5×10⁻⁴ 对比 |
| 标签平滑 | 交叉熵 + **标签平滑 0.1** | 软化 one-hot 标签，权重衰减仍为 5×10⁻⁴ |
| Mixup | 训练时对样本线性混合，**α = 0.2** | 混合两幅图像及其标签，损失按混合比例加权 |
| CutMix | 训练时裁剪粘贴图像块，**α = 1.0** | 局部替换式混合，同样按混合比例计算损失 |
| Dropout | 仅在分类头前引入， **Dropout 概率 0.5** | 结构正则，损失仍为交叉熵，权重衰减 5×10⁻⁴ |

下面特别说明标签平滑、Mixup 与 CutMix 在实验中的实现的形式与作用。

基线 CE 使用 one-hot 硬标签；下列三种方法在**标签或输入**上引入正则，损失仍写为 CE，但**训练目标与基线不同**。

**(1) 标签平滑（Label Smoothing，$\varepsilon=0.1$）**

对单个样本，记其真实类别为 $y$（$K=10$ 类）。将硬标签 $e_y$ 替换为软分布 $q$，对 $K=10$ 类：

$$
q_k = \begin{cases}
1-\varepsilon, & k=y, \\
\dfrac{\varepsilon}{K-1}, & k\neq y.
\end{cases}
$$

交叉熵改为 $\mathcal{L}_{\mathrm{LS}} = -\sum_{k=1}^{K} q_k \log p_k$，其中 $p_k$ 为 Softmax 概率。其含义是，不要求 $p_y\to 1$，允许非正确类保留小概率，抑制 logit 过度拉大。这一设置使得优化目标从 **拟合 one-hot** 变为 **拟合平滑分布**，loss 会高于基线模型，但是梯度会更温和。

**(2) Mixup（$\alpha=0.2$）**

符号约定如下：$\mathbf{x}$ 表示当前小批量中**第一张**参与混合的训练图像（32×32 RGB，经 1.1 节预处理后为张量）；$y \in \{1,\ldots,10\}$ 为其类别标签；$\mathbf{x}'$、$y'$ 分别为**同 batch 内随机抽取的另一张**图像及其类别；$\tilde{\mathbf{x}}$ 为对 $\mathbf{x}$ 与 $\mathbf{x}'$ 做像素级线性混合后、送入网络的混合图像；$\mathbf{z}$ 为网络对 $\tilde{\mathbf{x}}$ 前向得到的 logits 向量。

在上述约定下，采样 $\lambda \sim \mathrm{Beta}(\alpha,\alpha)$，构造

$$
\tilde{\mathbf{x}} = \lambda \mathbf{x} + (1-\lambda)\mathbf{x}'
$$
$$
\mathcal{L}_{\mathrm{Mixup}} = \lambda\,\mathrm{CE}(\mathbf{z}, y) + (1-\lambda)\,\mathrm{CE}(\mathbf{z}, y').
$$

这一操作在**整张图**上做像素线性混合，标签也按比例混合，迫使网络在样本之间的插值区域仍预测合理。其主要效果是，提供**数据层面**的增强与标签软组合，提高决策边界平滑性，减轻对个别样本的记忆。

需要特殊说明的是，每个 step 的**真实标签**为混合标签，**训练集准确率**按硬标签统计会系统性偏低，**不宜与基线 train acc 直接比较**，应以验证/测试准确率为准。

**(3) CutMix（$\alpha=1.0$）**

符号约定如下：$\mathbf{x}$、$y$ 与 $\mathbf{x}'$、$y'$ 分别表示待贴入区域的原图及其类别、提供替换块的配对图及其类别（含义同 Mixup）；$\tilde{\mathbf{x}}$ 为在 $\mathbf{x}$ 上裁入 $\mathbf{x}'$ 矩形块后得到的混合图像；$\mathbf{z}$ 为网络对 $\tilde{\mathbf{x}}$ 的 logits。混合系数 $\lambda$ 先由 Beta 分布采样，再按**被替换区域面积占整图比例**修正，用于加权下方两项交叉熵。

在上述约定下，与 Mixup 一样采样 $\lambda \sim \mathrm{Beta}(\alpha,\alpha)$，在 $\mathbf{x}$ 上随机取矩形区域，用 $\mathbf{x}'$ 的对应块替换，得到 $\tilde{\mathbf{x}}$；损失形式与 Mixup 相同：

$$
\mathcal{L}_{\mathrm{CutMix}} = \lambda\,\mathrm{CE}(\mathbf{z}, y) + (1-\lambda)\,\mathrm{CE}(\mathbf{z}, y').
$$

其含义是将混合控制在**局部区域**，更贴合实际图片样本中**目标只占图像一部分**的情形，常比全局 Mixup 保留更多局部纹理信息。

与 Mixup 类似，训练目标为混合标签；train acc 更低不代表模型更差。验证 loss 与基线 CE **可同图观察趋势**，但数值尺度会因 $\lambda$ 引入的随机性略有差异。

本实验的目的是，在固定网络宽度与训练超参(除正则化系数以外)下，探索不同正则化手段，如不同强度L2正则化、损失层面正则化(标签平滑)、数据层面正则化(Mixup/Cutmix)与结构层面正则化(Dropout)对网络性能的影响。

#### 1.6.2. 实验结果

##### 1.6.2.1. 不同强度L2正则化

下表为不同强度L2正则化消融实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| 基线 | 0.9318 | 183 | 0.9308 |
| 无权重衰减 | 0.9316 | 179 | 0.9297 |
| 强权重衰减 | 0.9260 | 162 | 0.9274 |

由表可知，关闭权重衰减（0.9297）与基线（0.9308）几乎相当，加强至 $10^{-3}$ 后性能明显下降（0.9274），说明本网络在 AdamW 下已受益于适度 L2，过强正则会欠拟合，基线 $5\times10^{-4}$ 较为合适。收敛速度上，强权重衰减最早（162），无权重衰减在 179 代，基线在 183 代达到最优性能；强 L2 虽更早达到验证峰值，但测试精度反而最低。相反，基线模型虽然最晚到达最优性能，但其测试精度最优。

下图为不同强度L2正则化消融实验中训练集与验证集的loss与accuracy曲线。

![1.6.2.1 不同强度L2正则化消融曲线](pic/1_6_2_1.png)

由图可知， 无权重衰减与基线曲线几乎重合，强权重衰减的验证 accuracy 明显偏低、验证 loss 偏高，与表中 0.9274 的降幅一致；曲线显示过强 L2 限制了模型充分拟合，支持表中**基线 wd 较为合适**的结论。

##### 1.6.2.2. 不同正则化方式

下表为不同正则化方式消融实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| 基线 | 0.9318 | 183 | 0.9308 |
| 标签平滑 | 0.9312 | 170 | 0.9329 |
| Mixup | 0.9470 | 200 | 0.9452 |
| CutMix | 0.9534 | 176 | 0.9480 |
| Dropout | 0.9338 | 179 | 0.9346 |

由表可知，Mixup 与 CutMix 带来最显著提升（测试 0.9452 / 0.9480），CutMix 验证集最优；标签平滑与 Dropout 相对基线仅有小幅改善。数据层面混合增强明显优于仅调整损失或分类头 Dropout，是本项目最有效的正则手段之一。收敛速度上，标签平滑（170）与 CutMix（176）较基线（183）更早达到验证峰值，表明这两种正则化手段能有效加速模型收敛；Dropout（179）与基线接近；而Mixup（200）则至训练结束验证精度仍在上升，表明当前训练预算下该实验组性能可能仍有进步空间。

下图为不同正则化方式消融实验中训练集与验证集的loss与accuracy曲线。

![1.6.2.2 不同正则化方式消融曲线](pic/1_6_2_2.png)

由图可知， CutMix 与 Mixup 的验证 accuracy 显著高于基线，CutMix 最终略优，对应表中 0.9480 / 0.9452；二者训练 accuracy 低于基线属混合标签所致，不宜据此判断欠拟合。标签平滑与 Dropout 曲线与基线接近，与表中小幅提升一致，图与表共同说明数据混合增强贡献最大。

### 1.7. 实验4：尝试不同的激活函数

#### 1.7.1. 实验设置

在 **1.3.1 基线** 基础上，**仅替换非线性激活函数**,其余设置均未改动。残差块内与入口层、块末输出处均使用同一激活函数类型。

| 实验 | 激活函数 | 说明 |
|------|----------|------|
| 基线（对照） | ReLU | 见 1.3.1，本组不再重复训练 |
| GELU | 高斯误差线性单元 | 平滑、非硬截断，常用于现代 CNN |
| Leaky ReLU | 负半轴斜率 **0.1** | 缓解神经元死亡，负值有小梯度 |

三种激活均加在卷积与残差块输出之后（入口层及每个块末）。下面给出三种激活函数在实验中的具体实现形式说明。记添加激活函数处标量为 $z$。

**(1) ReLU（基线）**

$$
\sigma(z)=\max(0,\,z).
$$

正值原样输出，负值置零；$z$ 长期为负时梯度为 0，可能出现神经元死亡。基线其余训练设置不变。

**(2) GELU**

$$
\sigma(z)=z\,\Phi(z),
$$

其中 $\Phi$ 为标准正态累积分布函数。在 0 附近平滑过渡，负值仍保留梯度，无 ReLU 的硬截断，常用于较新的卷积网络。

**(3) Leaky ReLU**

$$
\sigma(z)=\begin{cases} z, & z>0, \\ 0.1z, & z\le 0 \end{cases}
$$

负半轴斜率为 **0.1**用于在负区间仍传递梯度，缓解 ReLU 死亡问题。

本实验的目的是，在固定网络宽度与训练超参下，比较不同激活函数对收敛速度与最终分类误差的影响。

#### 1.7.2. 实验结果

下表给出激活函数消融实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| 基线（ReLU） | 0.9318 | 183 | 0.9308 |
| GELU | 0.9326 | 176 | 0.9329 |
| Leaky ReLU | 0.9320 | 189 | 0.9333 |

由表可知，应用三种激活函数最终测试准确率均在 0.9308–0.9333 之间，差异很小；GELU 与 Leaky ReLU 略优于 ReLU 基线，但提升幅度不足 0.2 个百分点，说明在该 ResNet 结构与训练配置下激活函数并非主要性能瓶颈。收敛速度上，GELU 最早（176），ReLU 基线为 183，Leaky ReLU 最晚（189）达到最优性能，收敛速度接近。综合来看，在当前实验设置下，换用不同的激活函数对收敛速度以及最优性能的影响有限。

下图为激活函数消融实验中训练集与验证集的loss与accuracy曲线。

![1.7.2 激活函数消融曲线](pic/1_7_2.png)

由图可知， 三条验证 accuracy 曲线中后期几乎重叠，GELU、Leaky ReLU 仅略在末端高于 ReLU，与表中 0.9329 / 0.9333 / 0.9308 的微小差距相符；loss 曲线形态亦相近，说明激活函数改动对训练动态影响有限。从训练集损失与准确率曲线来看，GELU 收敛快于 ReLU，ReLU 收敛快于 Leaky ReLU，这是因为 GELU 在零附近平滑可导，负值区仍保留梯度，ReLU 负半轴梯度为零会在早期略慢，Leaky ReLU 虽缓解死亡神经元但负斜率较小，等效更新幅度仍弱于 GELU。这与前文图表中达到验证集最优的训练代数先后顺序完全一致，呼应了图表解读中提及的、激活函数对最终精度影响有限但收敛速度仍有差异的判断。

### 1.8. 实验5：使用`torch.optim`尝试不同的优化器

#### 1.8.1. 实验设置

在 **1.3.1 基线** 基础上，**仅更换优化器类型及初始学习率**，其余设置与基线完全一致。

对每种优化器，在合理范围内各试 **3 个初始学习率**，调度仍为 **200 epoch 余弦退火**：

| 优化器 | 初始学习率 | 其余说明 |
|--------|------------|----------|
| **SGD** | 0.05、**0.1**、0.2 | 动量系数 **0.9** ，开启 Nesterov 动量 |
| **Adam** | 3×10⁻⁴、**1×10⁻³**、3×10⁻³ | 自适应学习率方法，与 AdamW 对比 decoupled 衰减差异 |
| **AdamW** | 3×10⁻⁴、1×10⁻³、3×10⁻³ | 其中 1×10⁻³ 即基线配置 |

需要特殊说明的是，三种优化器各试三个初始学习率，是因为同一数值的 lr 在不同算法下含义不同，需在各自常用量级内分别筛选。

对于**SGD（带动量 0.9、Nesterov）**，其更新幅度主要由全局 lr 决定，无逐参数自适应，在 CIFAR/ResNet 上通常取 0.1 附近，故表中 0.05、0.1、0.2 分别对应偏保守、常规与偏大的步长；相比之下 **Adam / AdamW** 会对各参数做自适应缩放，等效步长一般远小于 SGD@0.1，故采用 10⁻⁴～10⁻³ 量级的 3×10⁻⁴、1×10⁻³、3×10⁻³。因此，表中学习率按优化器分列，不共用同一套数字，最终优劣以验证/测试准确率为准。

本实验的目的是，在相同模型与损失下，比较不同一阶优化器搭配不同学习率尺度对训练稳定性与测试误差的影响。

#### 1.8.2. 实验结果

下表为优化器消融实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------------|--------|-----|-------|
| SGD lr=0.05 | 0.9386 | 137 | 0.9394 |
| SGD lr=0.1 | 0.9458 | 200 | 0.9465 |
| SGD lr=0.2 | 0.9440 | 176 | 0.9422 |
| Adam lr=3e-4 | 0.9276 | 189 | 0.9269 |
| Adam lr=1e-3 | 0.9392 | 189 | 0.9330 |
| Adam lr=3e-3 | 0.9230 | 183 | 0.9232 |
| AdamW lr=3e-4 | 0.9106 | 170 | 0.9130 |
| AdamW lr=1e-3 **(基线)** | 0.9318 | 183 | 0.9308 |
| AdamW lr=3e-3 | 0.9320 | 171 | 0.9328 |

由表可知，优化器与学习率影响非常显著：SGD（lr=0.1）测试最优（0.9465），明显优于 AdamW 基线（0.9308）；Adam 在 lr=1e-3 时次优（0.9330），lr 过大或过小均会退化；AdamW lr=3e-4 表现最差（0.9130）。说明对本 ResNet 而言，SGD+Nesterov 配合适当大 lr 优于默认 AdamW 配置。收敛速度上，SGD lr=0.05 最早（137），AdamW lr=3e-4 与 lr=3e-3 分别为 170、171，Adam 三组均在 183–189 代达到最优；SGD lr=0.1 的最优 epoch 为 200，说明验证精度在训练末期仍在上升，属于晚收敛、高上限的情形。

下图为优化器消融实验中训练集与验证集的loss与accuracy曲线。

![1.8.2 优化器消融曲线](pic/1_8_2.png)

由图可知，九组曲线清晰呈现**同一优化器下学习率尺度的影响**：**SGD** 中 **lr=0.1** 验证 accuracy 全程最高，至 epoch 200 仍在抬升，与表中测试 **0.9465** 一致；**lr=0.05** 前期上升更快但平台偏低（**0.9394**），**lr=0.2** 略逊且 loss 偶有震荡（**0.9422**）。**Adam** 以 **lr=1e-3** 最优（**0.9330**），**lr=3e-4** 收敛偏慢，**lr=3e-3** 后期明显回落（**0.9232**）。**AdamW** 三组整体低于 SGD，**lr=3e-4** 全程最差（**0.9130**），**lr=1e-3** 与 **lr=3e-3** 验证曲线接近（**0.9308 / 0.9328**）。不同优化器之间的性能比较需要选取每种优化器的最优学习率设置实验组进行观察，才能进一步得出结论。

下图为优化器消融实验中三种优化器各自验证集准确率最优实验组的训练集与验证集的loss与accuracy曲线。

![1.8.2 优化器最优实验组消融曲线](pic/1_8_2_best.png)

由图可知，在剔除各优化器内部次优 lr 的干扰后，**SGD lr=0.1**、**Adam lr=1e-3**、**AdamW lr=3e-3** 三条曲线的差距比全图更为直观：**SGD** 验证 accuracy 与 loss 仍全面领先，**Adam** 次之，**AdamW** 即使取组内验证最优配置，平台上限仍明显低于 SGD；loss 子图中 SGD 训练/验证同步降至更低水平，与上一张九组图中 **lr=0.1** 一枝独秀的现象相互印证。在当前为各优化器分别选取较优初始学习率的前提下，**SGD + Nesterov（lr=0.1）** 仍是三者中验证/测试表现最好的一组。这与 ResNet 在 CIFAR-10 上、配合 **200 epoch 余弦退火** 的训练方式相符：SGD 使用**全局统一的大步长**（0.1 量级），配合**动量 0.9 与 Nesterov** 在训练前期能沿较稳定的方向快速下降，后期随余弦调度步长自然减小，较易收敛到更低的 loss；Adam / AdamW 则对**每个参数自适应缩放**更新幅度，在相同训练轮次下等效步长通常远小于 SGD@0.1，对本实验中固定的网络宽度与权重衰减而言，探索效率不如 SGD 充分，故即使 Adam 取 **1e-3**、AdamW 取验证最优的 **3e-3**，验证与测试精度仍整体落后于 SGD@0.1。

最终结论是，在除优化器与学习率其余实验设置保持不变的情况下，**SGD + Nesterov（lr=0.1）** 为九组最优，测试 **0.9465**，较 **AdamW lr=1e-3** 基线（**0.9308**）提升约 **1.6** 个百分点；此外值得注意的是，优化器与学习率尺度的影响大于 Part 1 中损失函数、激活等改动。

### 1.9. 最优模型探索实验

#### 1.9.1. 实验设置

前文 1.4–1.8 节分别从通道宽度、损失与正则化、激活函数与优化器四方面做了消融：**宽网络**（1.4）、**CutMix / Mixup**（1.6.2.2）、**SGD + Nesterov**（1.8）均明显优于 **1.3** 基线（AdamW、默认宽度、无混合增强）。本小节在 **1.3.1** 的数据划分、200 epoch、无早停、余弦学习率与验证集选优流程不变的前提下，将上述因素**组合**训练，并与基线对照，以寻找 Part 1 上的较优配置。

共 **五组** 实验：**基线**（与 1.3 相同，作对照）及四条 **combine** 配方；各组在网络宽度、优化器与学习率、正则化三方面的配置如下表（通道数写法与 1.4 节一致，括号内为首阶段宽度）。

| 实验 | 网络大小 | 优化器 + 学习率 | 正则化 |
|------|----------|-----------------|--------|
| 基线 | 基线（64）64–128–256–512 | AdamW，lr=1×10⁻³ | L2，wd=5×10⁻⁴；无 Mixup/CutMix |
| 实验1 | 宽网络（96）96–192–384–768 | SGD（Nesterov，动量 0.9），lr=0.1 | L2，wd=5×10⁻⁴；CutMix，α=1.0 |
| 实验2 | 宽网络（96）96–192–384–768 | SGD（Nesterov，动量 0.9），lr=0.05 | L2，wd=5×10⁻⁴；CutMix，α=1.0 |
| 实验3 | 基线（64）64–128–256–512 | SGD（Nesterov，动量 0.9），lr=0.1 | L2，wd=5×10⁻⁴；CutMix，α=1.0 |
| 实验4 | 基线（64）64–128–256–512 | SGD（Nesterov，动量 0.9），lr=0.1 | L2，wd=5×10⁻⁴；Mixup，α=0.2 |

**基线** 行与 **1.3** 相同，而**实验1–4** 为不同优化路径的 combine 配方：**实验1** 将 1.4 / 1.6.2.2 / 1.8 中的较优单项(宽网络/SGD+lr=0.1/cutmix)叠加；**实验2** 使用了 1.8 中使用 SGD 的次优学习率(0.05),其余设置与实验1一致；**实验3** 与 **实验4** 则在基线网络宽度下对照 CutMix 与 Mixup，与实验1使用相同的学习率与优化器配方。除表中列出的 optimizer、学习率、网络宽度(通道数)与正则化差异外，各实验超参与基线一致。

本实验的目的是，在统一训练流程下验证**多有利因素叠加**能否超过单项消融的最优测试精度，并确定 Part 1 的最优综合配置。

#### 1.9.2. 实验结果

下表为基线与四条 combine 配方实验结果汇总。

| 实验 | 参数量 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|--------|------------------|------------|--------------|
| 基线 | 11.17M | 0.9318 | 183 | 0.9308 |
| 实验1 | 25.13M | **0.9600** | 189 | **0.9608** |
| 实验2 | 25.13M | 0.9558 | 184 | 0.9527 |
| 实验3 | 11.17M | 0.9566 | 187 | 0.9569 |
| 实验4 | 11.17M | 0.9506 | 193 | 0.9528 |

由表可知，**实验1**（宽网络（96）+ SGD lr=0.1 + CutMix）测试最优（**0.9608**），较 **基线** 提升约 **3.0** 个百分点，亦高于 1.8 节单项 SGD@0.1（0.9465）、1.6.2.2 单项 CutMix（0.9480）与 1.4 节宽网络（0.9346）；**实验1** 与 **实验2** 在宽网络与 CutMix 已固定时仅差学习率，lr=0.05 时测试降至 0.9527，说明 lr=0.1 仍然是更合适的学习率取值；**实验3** 在 **基线（64）** 宽度下 SGD+CutMix 仍达 0.9569，说明优化器与数据增强贡献显著，但其性能不及实验1，证明宽网络能够为模型性能带来显著增益；**实验4** 的 Mixup（0.9528）略低于 **实验3** 的 CutMix，则体现了在当前超参数设定下 Mixup 不及 CutMix 有效。综合看，**宽网络、CutMix 与 SGD lr=0.1 的叠加**为本任务 Part 1 的最优组合。收敛速度上，实验2 最早（184），实验1 为 189，实验3 为 187，基线为 183，实验4 最晚（193）达到最优。这说明，各 combine 组达到验证峰值的早晚差异不大，精度差距主要来自结构与训练策略而非收敛先后。

下图为基线与四条 combine 配方在训练集与验证集上的 loss 与 accuracy 曲线。

![1.9.2 最优模型探索曲线](pic/1_9_2.png)

由图可知，本实验对比中唯一使用AdamW优化器的 **基线** 训练曲线收敛最快：约 epoch 75 后训练 loss 已接近 0、训练 accuracy 贴近 1.0，但验证 loss 稳定在约 0.55、验证 accuracy 平台在 0.92–0.93，train/val 差距明显，与 1.3.2 中**拟合充分而泛化一般**的形态一致。

**实验1–3**三者走势高度相近：训练端 loss 与 accuracy 均明显慢于基线（末期训练 accuracy 约 0.75、训练 loss 约 0.6），验证端却整体优于基线——验证 loss 可降至约 0.15–0.2、验证 accuracy 升至约 0.95，且多数阶段出现**验证优于训练**（虚线 accuracy 高于实线、验证 loss 低于训练 loss），这是 CutMix 混合标签下训练 accuracy 按硬标签统计所致。

三者中, **实验1** 验证 accuracy 平台上限略高、后期更稳，与表中测试 **0.9608** 最优一致；**实验2** 全程紧贴 **实验1** 但末端验证 accuracy 略低，验证了使用SGD优化器时lr取值0.1相较于0.05更好；**实验3** 全过程中性能始终略微不如 **实验1**，验证了在 CutMix+SGD 已固定时增宽网络仍有小幅增益。

**实验4**训练曲线介于基线与 **实验1–3** 之间，但验证 loss/accuracy 波动更大、平台约 0.92，泛化不及 CutMix 三组，与表中 **0.9528** 及 **实验3** 的 **0.9569** 相符，印证了 CutMix 数据增强的性能增益大于 Mixup 的结论。

综上，combine 配方的主要收益体现在**验证/测试泛化**，而非训练集拟合速度；**实验1** 为五组中最优。

### 1.10. 网络内在原理分析

本节从**错误结构、典型错例关注区域、特征表征与损失波动**四个角度解释最优模型为何有效。为避免指代歧义，本文统一约定：**基线模型**指 1.3 节配置（AdamW + 64-128-256-512），**最优模型**指 1.9 节实验1（宽网络 + CutMix + SGD lr=0.1）。除卷积核对比图外，本节行为分析图默认来自最优模型 checkpoint。

#### 1.10.1. 最优模型错误结构与典型错例解释

下图先给出**最优模型（1.9 实验1）**在测试集上的混淆矩阵（按真实类别归一化），用于观察全局错误分布。

![1.10.1 混淆矩阵](pic/1_10_1_confmat.png)

可见矩阵主对角线占主导，说明模型整体分类稳定。非对角线中的主要错误也较集中：`cat→dog` 为 **62**、`dog→cat` 为 **35**，`truck→automobile` 为 **22**，`airplane→ship` 为 **16**，`bird→frog` 为 **16**，`horse→dog` 为 **14**，`deer→cat` 为 **13**。这些主要误差基本都发生在外观纹理或轮廓相近的类别之间，符合 CIFAR-10 低分辨率条件下的常见混淆模式。该图回答了**模型主要错在何处**这一问题。

进一步选取测试集中**最高置信度的 Top-10 错误样本**，观察最优模型在**高置信但误判**情形下的行为。

![1.10.1 Top-10 高置信错误](pic/1_10_1_top10_errors.png)

这些样本普遍具有主体尺度小、局部遮挡、姿态极端或背景干扰强等特点。模型在困难样本上的错误并非随机分布，而是集中发生在**局部纹理相似但整体语义不同**的图像上。

最后对以上Top-10高置信错误样本叠加 Grad-CAM 热力图。

![1.10.1 Grad-CAM 错例热力图](pic/1_10_1_gradcam.png)

结合原图与热力图，可按热区形态列举五例：

- `idx=7099`（`cat→dog`, conf=0.984）：原图中主体较小且背景纹理明显；热区在目标周围大范围扩散，并覆盖背景高响应区域。
- `idx=6646`（`dog→cat`, conf=0.986）：原图可见犬类主体，但局部毛发与边缘对比强；热区主要集中在局部高对比块，未形成对完整犬体轮廓的连续覆盖。
- `idx=577`（`truck→ship`, conf=0.979）：原图目标较小、远景占比高；热区对目标与周边区域同时激活，对车辆关键结构关注不足。
- `idx=7590`（`bird→deer`, conf=0.980）：原图中主体颜色与背景接近，鸟类关键部件不突出；热区偏向主体大块纹理而非判别性部件。
- `idx=2895`（`horse→dog`, conf=0.983）：原图中目标姿态特殊，马的典型长体结构不明显；热区集中在头部局部区域，缺少对整体形态的覆盖。

上述错例的热区多集中在局部区域或含明显背景响应，与混淆矩阵中外观相近类对的主要非对角线错误一致。结合 `32×32` 分辨率与网络下采样、全局平均池化，难样本上更易出现**高置信但局部热区驱动**的误判。

下一小节对 1.3 基线与 1.9 实验1 最优模型的卷积核进行对比。

#### 1.10.2. 最优模型与基线模型卷积核可视化

本小节比较**1.3 基线模型**与**1.9 实验1最优模型**在两个关键位置的卷积核：其一是 1.2.2 节入口层（Stem）的 `conv1`，即从 RGB 输入映射到首层特征图的第一层 3×3 卷积；其二是 1.2.3 节残差阶段 3 的首个残差块中第一层 3×3 卷积 `layer3.0.conv1`。选择这两层的原因是，`conv1` 对应低层视觉基元提取（边缘、颜色对比与粗纹理），`layer3.0.conv1` 对应中层局部语义模式组合；二者与 1.2 中**入口层 -> 残差阶段递进表征**的叙述一致，能够覆盖从低层到中层的关键特征提取链路。

![1.10.2 卷积核可视化](pic/1_10_2_kernels.png)

从卷积核本身的特性来看，无论是基线模型还是最优模型，其在 `conv1` 层可以观察到清晰的方向性边缘核、颜色对比核与低频平滑核，这说明模型确实学习到了图像分类所需的基础视觉原语（边缘、色彩差异、粗纹理轮廓）。在 `layer3.0.conv1` 中层，卷积核呈现更强的纹理组合与局部形状模式，说明网络已从低层的像素级对比，过渡到更接近语义部件的中层表示。这一层级演化符合深层卷积网络的典型表征规律：**浅层提取通用视觉基元，中层组合形成可判别模式**。

进一步做两模型的逐层对照：在 `conv1` 与 `layer3.0.conv1` 上，两模型整体均呈边缘与纹理类模式，**逐核对照时肉眼差异有限**，不宜归纳为最优模型方向覆盖更完整或中层响应更稳定。图中为可视化选取的部分通道；最优模型通道更宽（96–192–384–768 对比 64–128–256–512），权重分布不同属结构差异所致，与测试精度提升的关联仍以 1.9 实验结果为主。

1.10.3 给出损失曲面包络。

#### 1.10.3. 损失曲面包络

本小节仍以 1.9 最优路径（宽网络 + CutMix + SGD）为主体，考察在该配置下不同学习率（`0.05, 0.1, 0.15, 0.2`）对应的 step-loss 包络，从而评估优化过程对步长扰动的稳定性。值得一提的是，本验证试验在上述设置的基础上，额外加入下图的 No-CutMix 版本作为辅助，用于排除 CutMix 额外随机混合带来的噪声干扰，从而更有力地验证从带 CutMix 实验对照中观察到的趋势是否可靠。出于训练成本考虑，八组实验均将训练代数从1.9中最优模型的200代降低到50代，并在同一步索引上取损失数值的最大值/最小值曲线形成包络。另外，出于绘图美观性考虑，图片中每8个step记录一个点，并添加了长度为9的滑动平均平滑。

![1.10.3 损失曲面包络](pic/1_10_3_loss_landscape_sgd.png)

从图中可见，**上图 CutMix 包络**在训练早期明显更宽，随后随 step 增加持续收窄，并在中后期进入相对稳定区间；**下图 No-CutMix 包络**也呈现相同的前宽后窄趋势，但整体下降更快、数值更低。两幅子图在趋势层面一致，说明学习率扰动在训练前期更敏感、在训练后期逐步被吸收，包络收敛是该组实验中稳定出现的动力学现象。

进一步看，添加 CutMix 的组别在每个 batch 引入区域替换与混合标签，使优化目标更复杂，因此上图在同等 step 下保留了更高损失带以及更剧烈的局部波动；不添加 CutMix 的组别则用于对照，说明包络收敛趋势并非仅由 CutMix 的随机性单独造成。

### 1.11. 总结

在 **1.3** 基线实验中，CifarResNet 于 AdamW（lr=1×10⁻³）、200 epoch 余弦退火与权重衰减 5×10⁻⁴ 下测试准确率为 **0.9308**，验证与测试接近、后期曲线未见明显验证恶化，可作为 Part 1 各组消融的统一对照。**1.4** 宽度实验表明，通道由 64 增至 96 可提升至 **0.9346** 并略早达到最优 epoch，收窄至 32 则降至 **0.9107**；容量增大有益，但单项增宽带来的提升有限，尚不足以解释后续更高测试精度。

**1.5** 与 **1.6** 从**损失目标与正则化**侧展开：Focal、Multi-Margin 未稳定优于交叉熵（Focal 测试 **0.9236**），说明在本结构下替换损失并非主路径；L2 强度以基线 wd=5×10⁻⁴ 较合适，过强衰减会限制拟合。在多种正则手段中，**CutMix / Mixup** 提升最为突出（测试 **0.9480 / 0.9452**），明显高于标签平滑与 Dropout 的边际改动；数据层面混合增强是除优化器外最有效的正则手段之一，且训练 accuracy 不宜与基线直接比较。

**1.7** 激活函数消融中，GELU、Leaky ReLU 与 ReLU 的差异仅在约 **0.2** 个百分点量级；**1.8** 优化器实验则显示影响显著——**SGD + Nesterov（lr=0.1）** 测试 **0.9465**，较 AdamW 基线高约 **1.6** 个百分点，亦优于各组 Adam/AdamW。曲线与表格共同表明：在本 ResNet 与 CIFAR-10 设定下，**优化器与学习率尺度** 的优先级高于损失形式与激活细节。

**1.9** 将 1.4–1.8 中的较优因素组合训练：**实验1**（宽网络 + CutMix + SGD lr=0.1）测试集准确率达 **0.9608**，超过任一单项消融最优；**实验3** 在不增宽网络仅替换优化器并添加 CutMix 时测试集准确率仍达 **0.9569**，说明改用SGD优化器与 CutMix 可独立对模型性能产生正向影响；**1.9.2** loss与accuracy曲线图中各 combine 实验组与基线模型曲线的对比进一步表明 combine 组的主要收益体现在**验证/测试泛化**，而非训练集拟合速度。

**1.10** 补充了 **1.9 实验1** 的错例与 Grad-CAM、与基线的卷积核对比，以及不同学习率下的 loss 包络；主要误差仍集中在局部纹理相近类别。**1.10.3** 在四种学习率下观察到包络由前宽后窄，训练后期对步长扰动更不敏感。

综上，Part 1 以 1.3 的 AdamW 基线（约 93.1%）为对照，经 1.4–1.8 消融可见：在当前超参数设置下，更换损失函数、加强L2正则、添加 label smoothing 或更换激活函数带来的提升有限；而加宽卷积通道、对训练集添加 CutMix/Mixup 与更换更合适的优化器学习率组合(SGD + lr=0.1)的提升较为显著。1.9 将三者与宽网络组合后，**实验1**（宽网络 96–192–384–768 + SGD lr=0.1 + CutMix）测试误差为 **3.92%**（测试集准确率 **96.08%**），为第一部分最优。


## 2. 批归一化

### 2.1. 数据集使用说明

本任务同样使用 CIFAR-10。关于数据集的官方划分、训练集与验证集的 **9 : 1** 划分（随机种子 **2020**）、数据预处理，以及测试集仅在训练结束后评估一次等约定，**均与第一部分相同**，不再赘述。唯一有区别的是对训练集的使用方式：除了仍然用于参数更新外，在各个实验中，训练损失以及优化相关量按每个小批次而非每个epoch进行记录，方便后续损失曲面与梯度分析。

### 2.2. 网络结构：VGG-A 与 VGG-A + BN

本节采用课程提供的 **VGG-A** 卷积网络（共 **11 个含权重的层**：特征提取部分 8 个卷积层 + 分类头 3 个全连接层），并按 CIFAR-10 的 **32×32** 输入对原 ImageNet 版 VGG-A 的全连接尺寸做了缩减。下文先说明无 BN 的 **VGG-A** 拓扑；**VGG-A + BN** 在相同拓扑上于指定位置插入批归一化，结构差异集中在 2.2.2 节。

#### 2.2.1. VGG-A

##### 2.2.1.1. 整体数据流与特征图尺寸

输入一张 32×32×3 的图像后，网络由 **特征提取模块** 与 **分类头** 串联而成。特征提取部分由五个阶段组成，每阶段在若干 3×3 卷积与非线性激活之后，以 **2×2、步长 2 的最大池化** 将空间分辨率减半；五个阶段依次将边长由 32 降至 16、8、4、2，最终在 **1×1** 空间尺寸上得到 **512 通道** 的特征图。分类头将该 512 维向量经两层隐层全连接（各 512 维、ReLU）映射到 10 类 logits。各阶段通道与空间尺寸如下：

| 阶段 | 卷积配置（输入→输出通道） | 池化后空间尺寸 | 输出通道数 |
|------|---------------------------|----------------|------------|
| 1 | 3 → 64（1 层 3×3 卷积） | 16×16 | 64 |
| 2 | 64 → 128（1 层） | 8×8 | 128 |
| 3 | 128 → 256，256 → 256（2 层） | 4×4 | 256 |
| 4 | 256 → 512，512 → 512（2 层） | 2×2 | 512 |
| 5 | 512 → 512，512 → 512（2 层） | 1×1 | 512 |
| 分类头 | 512 → 512 → 512 → 10（3 层全连接） | — | 10 |

各卷积层均采用 **3×3 核、步长 1、四周填充 1 像素**，在不改变空间分辨率的前提下堆叠通道；**ReLU** 置于每个卷积输出之后、池化之前。与标准 VGG-A 一致，阶段 3–5 各含 **两个** 连续卷积块再下采样，阶段 1–2 各仅 **一个** 卷积块，从而在参数量可控的前提下逐层加深表征。

##### 2.2.1.2. 特征提取模块

**阶段 1–2** 分别完成从 RGB 到 64、再到 128 通道的初步映射，并各经一次最大池化，侧重边缘与简单纹理。**阶段 3** 在 8×8 分辨率上通过两层 256 通道卷积提取更丰富的局部模式后下采样至 4×4。**阶段 4、5** 在更小空间尺寸上堆叠 512 通道卷积，形成高层语义特征；阶段 5 输出 1×1×512 的特征图，等价于对每个样本得到一个长度为 512 的全局特征向量（由五次池化自然得到，无需额外全局池化层）。

##### 2.2.1.3. 分类头

将 512 维特征向量依次通过 **512 → 512 → 10** 的三层全连接；前两层后接 ReLU，最后一层输出 10 维 logits，训练时对输出施加交叉熵损失。分类头 **不使用** Dropout 或批归一化。

#### 2.2.2. VGG-A + BN

##### 2.2.2.1. BN 的插入位置

**VGG-A + BN** 与 2.2.1 节网络在阶段划分、卷积核尺寸、通道数、池化位置及分类头拓扑上 **完全一致**；差别仅在于特征提取模块中，在 **每一个 3×3 卷积层之后、ReLU 之前** 插入一层 **二维批归一化**，共 **8 处**（对应八个卷积层）。**分类头的三层全连接及其间的 ReLU 均不添加 BN**，以保持与无 BN 基线在分类头上的可比性。

##### 2.2.2.2. 批归一化的数学形式

单个卷积–BN–ReLU 子块中，设卷积输出（BN 的输入）为四维张量 $I_{b,c,x,y}$，其中 $b$ 为小批量内样本索引，$c$ 为通道，$(x,y)$ 为空间位置。BN 对该通道在 **当前小批量、该通道所有空间位置** 上的激活做归一化，并施加可学习的通道级仿射变换，输出为 $O_{b,c,x,y}$：

$$
O_{b,c,x,y} \leftarrow \gamma_{c}\,\frac{I_{b,c,x,y}-\mu_{c}}{\sqrt{\sigma_{c}^{2}+\epsilon}}+\beta_{c}
\quad \forall\, b,c,x,y
$$

其中 $\mu_{c}$、$\sigma_{c}^{2}$ 分别为通道 $c$ 在当前小批量上的均值与方差：

$$
\mu_{c}=\frac{1}{|B|}\sum_{b,x,y} I_{b,c,x,y},\qquad
\sigma_{c}^{2}=\frac{1}{|B|}\sum_{b,x,y}\bigl(I_{b,c,x,y}-\mu_{c}\bigr)^{2}
$$

$B$ 表示该小批量中通道 $c$ 的全部激活取值；$\epsilon$ 为数值稳定项；$\gamma_{c}$、$\beta_{c}$ 为训练中学习的缩放与平移参数。**推理阶段** 使用训练过程中积累的通道均值、方差的滑动估计代替上式中的小批量统计量。归一化后再经 ReLU，随后进入各阶段末尾的 2×2 最大池化，与无 BN 版本的下采样节奏相同。

从作用上看，BN 将每层卷积输出在通道维度上拉向零均值、单位方差附近的分布，再经 $\gamma_{c}$、$\beta_{c}$ 恢复表达能力，从而缓解内部协变量偏移、使损失曲面更平滑，有利于后续 2.3、2.4、2.5 节中的训练对比与优化 landscape 分析。

### 2.3. 基线实验：VGG-A 与 VGG-A+BN 对比实验

#### 2.3.1. 实验设置

本小节给出 **2.3 基线对比实验** 的完整训练配置。数据划分与预处理同 **2.1** 节；网络拓扑分别为 **2.2.1** 节的 VGG-A 与 **2.2.2** 节的 VGG-A + BN。除是否在特征提取模块中插入 BN 外，两组实验 **共享同一套超参数与训练流程**，以便比较 BN 对分类性能与训练过程的影响。

训练设置上，训练批量为 512，固定 200 个 epoch、不使用早停（与第一部分一致），随机种子 2020；需要指出，除每 epoch 末记录训练/验证损失与准确率外，还在每个小批量更新后记录**训练损失、学习率、全局梯度范数与参数更新范数**，供后续分析，且本组不启用沿梯度方向的距离扫描（见 2.5 节）；损失为交叉熵，不用标签平滑、Mixup 或 CutMix；优化器为 AdamW，初始学习率 1×10⁻³，权重衰减 5×10⁻⁴（decoupled）；学习率按余弦退火在 200 个 epoch 内从 1×10⁻³ 衰减至接近 0，每 epoch 末更新；训练流程上，每 epoch 在验证集上选验证准确率最高的权重，训满 200 epoch 后各在测试集上评估一次，得到 VGG-A 与 VGG-A + BN 的最终指标。

本实验的目的是，在相同训练配置下对比 VGG-A 与 VGG-A + BN，检验批归一化对分类性能与训练稳定性的影响，并为 2.4 节损失曲面实验与 2.5 节梯度实验提供**有无 BN** 的性能基线；同一配置下的最优权重亦用于 **2.3.3** 的中间层特征与激活统计对比。

#### 2.3.2. 实验结果

下表给出VGG-A与VGG-A+BN对比实验结果汇总。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| VGG-A（无 BN） | 0.8662 | 175 | 0.8693 |
| VGG-A + BN | 0.9038 | 170 | 0.9006 |

由表可知，插入 BN 后验证/测试准确率分别提升约 3.7 与 3.1 个百分点（0.8693 → 0.9006），且 **+BN 在 epoch 170 达到验证峰值、无 BN 为 epoch 175**，收敛更快，表明 BN 在 VGG-A 上同时改善了收敛速度与泛化；该结果为 2.4、2.5 节的 landscape 分析提供了性能层面的对照基线。

下图为VGG-A与VGG-A+BN对比实验中训练集与验证集的loss与accuracy曲线。

![2.3.2 BN 对比曲线](pic/2_3_2.png)

由图可知， VGG-A + BN 的验证 accuracy 上升更快、平台更高，无 BN 模型在训练后期验证 loss 抬升而训练 loss 仍降，呈现过拟合；加 BN 后 train/val 更一致，与表中测试 0.9006 vs. 0.8693 及 +BN 在 epoch 170 达到验证峰值、无 BN 为 epoch 175 相吻合，从曲线侧验证了 BN 对收敛与泛化的双重改善。从训练集损失与准确率曲线来看，带有 BN 组收敛速度显著快于不带 BN 组，这是因为 BN 稳定各层激活分布，使每层输入尺度更易处于合适区间，前向与反向传播中有效梯度更易传递，因而训练 loss 与 accuracy 更早进入较优区间。这与表中 +BN 于 epoch 170、无 BN 于 epoch 175 达到验证峰值及测试准确率 0.9006 对 0.8693 的提升相一致，从训练与验证两侧共同印证 BN 对收敛速度的加速效果。

#### 2.3.3. 特征表征对比

##### 2.3.3.1. 实验设置

2.3.2 已从**性能**（准确率/损失曲线）说明 BN 的收益；本小节补充 **特征** 层面的对比。加载 **2.3** 节 VGG-A 与 VGG-A+BN 的验证最优权重（`best.pt`），数据预处理与 **2.1** 一致：特征图在**测试集**上选取 **VGG-A** 的 Top-4 高置信错分样本（两模型使用**相同输入**）；激活统计在**整个验证集**（约 5,000 张）上批量前向得到。

本实验的操作流程是，从前序实验训练好的 VGG-A 与 VGG-A+BN 的最优网络中提取 **Stage 1 / Stage 3** 池化后特征图（空间尺寸分别为 16×16×64 与 4×4×256），可视化时对通道取均值得到 2D 热力图；同时在 **Stage 1 / Stage 3 / Stage 5**（1×1×512）统计激活绝对值均值、全局标准差、通道标准差均值与 ReLU 后正值比例。

##### 2.3.3.2. 实验结果

下表为两模型在验证集上的层间激活统计汇总（Stage 为池化后语义阶段）。

| 阶段 | 模型 | 激活绝对值均值 | 全局标准差 | 通道标准差均值 | 正值比例 |
|------|------|----------------|------------|----------------|----------|
| S1 | VGG-A | 0.0401 | 0.0749 | 0.0579 | 0.465 |
| S1 | VGG-A+BN | 0.4272 | 0.6247 | 0.4948 | 0.640 |
| S3 | VGG-A | 0.0355 | 0.1469 | 0.0631 | 0.105 |
| S3 | VGG-A+BN | 0.2861 | 0.4673 | 0.3977 | 0.466 |
| S5 | VGG-A | 0.5362 | 2.2505 | 1.2338 | 0.163 |
| S5 | VGG-A+BN | 0.7213 | 1.1324 | 1.0781 | 0.612 |

由表可知，无 BN 模型在 **S3** 的正值比例仅 **0.105**，至 **S5** 亦仅 **0.163**；而 +BN 在 S3/S5 分别为 **0.466** 与 **0.612**。表象上是中间层与深层 ReLU 输出更**稀疏**；机制上，这与无 BN 时**层间激活尺度随深度漂移**直接相关：每一层卷积输出 $I_{b,c,x,y}$ 的均值与方差取决于当前权重及上一层 ReLU 后的非负、非零均值分布，经三次池化后，落入 ReLU 负半轴的通道比例累积上升，形成大量**永久为零**的通道（dying ReLU）。+BN 则在每个 Conv 之后按 2.2.2 节式 $(I-\mu_c)/\sqrt{\sigma_c^2+\epsilon}$ 将通道拉向零均值、单位方差，再经可学习的 $\gamma_c,\beta_c$ 恢复表达力，使进入 ReLU 的预激活更常处于**有效梯度区间** $(0,\infty)$，故正值比例显著升高。

表中 S1/S3 处 +BN 的**全局标准差、通道标准差均值反而更大**，但这一现象并不意味着带有BN的网络**更不稳定**：BN 的训练目标正是把各通道方差**规范化后再仿射重标定**，因此池化后统计量高于无 BN（S1 通道 std 约 **0.49** vs **0.06**）符合机制预期。更关键的对比在 **S3 直方图形态** 与 **S5 全局 std**：无 BN 在 S3 近零尖峰对应**信息坍缩**；至 S5 无 BN 全局 std 升至 **2.25** 而 +BN 为 **1.13**，说明无 BN 网络在深层仍保留少数幅度极大的通道、整体分布**长尾更重**，而 +BN 通过逐层重参数化抑制了深层激活的失控放大，使 512 维语义向量各通道贡献更均衡。

下图为 VGG-A 测试集 Top-4 高置信错分样本上的特征图对比（通道均值）；每行同一输入，列为原图、两模型在 S1/S3 的并排热力图。

![2.3.3 特征图对比](pic/2_3_3_feature_maps.png)

所选样本均为 VGG-A 置信度 **1.000** 的错例（如 idx=147 **bird→frog**、idx=312 **ship→airplane** 等）。S1 上两模型均在物体轮廓附近有相近高响应，+BN 在均匀背景区通道均值略高，与表中 S1 激活尺度更大一致。S3 仅为 **4×4** 通道均值图，四例之间模式并不一致，**不宜**由个别错例归纳无 BN 背景虚高或 +BN 更稀疏聚焦；中层差异应以验证集统计（正值比例、下节直方图）为主。

下图为验证集上的激活统计：左图为各阶段通道标准差均值，右栏为 **S3** 激活值密度分布。

![2.3.3 激活统计](pic/2_3_3_activation_stats.png)

由右栏可知，**S3** 处无 BN 约 **90%** 激活为 0（右上），+BN 约 **55%**；右下仅对 **>0** 激活作直方图，+BN 非零分布更宽、无 BN 更贴近 0。这意味着无 BN 时大量 ReLU 输出被截断为 0、可用于反向传播的非零激活偏少且幅度偏小，而 +BN 在 S3 保留了更多幅度适中的非零响应，与上文表中 S3 **正值比例** 及 2.3.2 的性能差异相一致。左图至 S5 两模型通道 std 均上升，但 +BN 全局波动更小。从优化机理看，BN 对损失曲面的平滑作用（2.4）与梯度沿距离变化更温和（2.5）并非仅来自**输出端准确率**，而是源于前向链路上**雅可比更可控**：当各层输入尺度稳定、ReLU 死神经元较少时，$\partial L/\partial W$ 不会因个别通道的极端激活而剧烈震荡，一阶优化器在固定步长下更安全。换言之，2.3.3 的表征证据与后文中 2.4–2.5 的损失曲面与梯度场证据构成了一致的因果链：**BN 重参数化显著提高了层间激活可利用率，从而使得有效梯度传递更稳定，进一步同时显著改善了训练与泛化性能**。

##### 2.3.3.3. 小结

综合表、特征图与激活分布，BN 在 VGG-A 上的收益可概括为三层：从**信号层**来看，BN有效缓解了内部协变量偏移，降低 ReLU 链式截断导致的中间层信息坍缩；从**表征层**来看，中层特征由**分散、背景敏感**转为**更聚焦、通道分工更明确**；最后从**优化层**来看，为 2.4 更窄的 loss 包络与 2.5 更小的梯度差提供有利的前置条件。2.3.2 回答**BN 是否更好**，本节回答**BN 在特征提取阶段如何更好**，二者相辅相成，共同回答了带 BN 的 VGG-A，相较于无 BN 的 VGG-A 在性能与特征角度的显著优势，为后续分析提供了坚实基础。

### 2.4. 损失曲面实验

#### 2.4.1. 实验设置

本实验在 **2.3.1** 基础上进行，**除训练轮次、学习率及其调度方式外，其余设置均与 2.3.1 完全一致**。具体变动如下：

- 最大训练轮次为 **100** 而非基线的 200 代；

- 学习率从集合 $\{1\times10^{-3},\,2\times10^{-3},\,1\times10^{-4},\,5\times10^{-4}\}$ 中取；

- 对 **VGG-A** 与 **VGG-A + BN** 不再使用余弦退火学习率调度，改用**恒定学习率**，四种学习率、两类网络共 **八组** 独立训练。

本小节只考察作业要求中的第一项指标：**损失曲面或损失值的变化**。做法为：对每组固定学习率单独训练，在相同的训练步索引 $t$ 上比较各次运行的 train loss，取同一步的极大、极小值构成包络，用以反映**在相同训练进度下、不同步长设定所引起的损失高低差**。

需要特别解释的是不再使用学习率调度的原因。包络所要刻画的是步长因素对损失的影响；若采用余弦退火，有效步长会随时间变化，同一步索引上的差异将混入调度阶段的不同，无法单独衡量损失对步长的敏感程度。故本实验在固定学习率下以多种 $\eta$ 代表不同优化尺度；梯度沿距离的变化见 **2.5 节**。

本实验的目的是，比较有 BN 与无 BN 模型在训练过程中、同一步索引上 train loss 的波动范围，观察 BN 是否使损失曲面更平滑、对步长更不敏感；可视化结果见 2.4.2。

#### 2.4.2. 实验结果

下表为损失曲面实验结果汇总。

| 实验 | 学习率 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|--------|------------------|------------|--------------|
| VGG-A（无 BN） | 1×10⁻³ | 0.8522 | 87 | 0.8507 |
| VGG-A（无 BN） | 2×10⁻³ | 0.8380 | 96 | 0.8475 |
| VGG-A（无 BN） | 1×10⁻⁴ | 0.8274 | 88 | 0.8275 |
| VGG-A（无 BN） | 5×10⁻⁴ | 0.8548 | 75 | 0.8584 |
| VGG-A + BN | 1×10⁻³ | 0.8888 | 90 | 0.8858 |
| VGG-A + BN | 2×10⁻³ | 0.8910 | 87 | 0.8845 |
| VGG-A + BN | 1×10⁻⁴ | 0.8420 | 84 | 0.8415 |
| VGG-A + BN | 5×10⁻⁴ | 0.8884 | 98 | 0.8831 |

由表可知，在四种固定学习率下，VGG-A + BN 的测试准确率始终高于无 BN 模型（约 0.84–0.89 vs. 0.83–0.86）；无 BN 最优出现在 lr=5×10⁻⁴（0.8584），加 BN 后在 lr=2×10⁻³ 达到最高验证 0.8910。BN 不仅抬高整体性能，也使不同步长下的表现更集中，为损失曲面更平滑提供了数值佐证。

下图为四种学习率下 VGG-A 与 VGG-A + BN 在训练全程中的 loss 上下界包络；绘图为每 11 个 mini-batch 取一点（约每 epoch 8 点），训练日志中则逐步记录 loss。

![2.4.2 损失曲面对比](pic/2_4_2_loss_landscape.png)

由图可知，绿色（无 BN）包络带宽且整体偏高，红色（+BN）包络更窄、更低，训练中后期差距尤为明显；说明同一步索引上、不同学习率引起的 loss 波动更小，与表中 BN 在各 lr 下测试准确率更高一致，从损失值侧支持 BN 使 landscape 更平滑、对步长更不敏感的结论。

### 2.5. 梯度实验

#### 2.5.1. 实验设置

本实验用于衡量梯度预测性与梯度随距离的最大差值，实验设定与 2.4.1 基本保持一致，下列仅列出差异。

- 共 **两组** 训练：**VGG-A** 与 **VGG-A + BN**，学习率固定为 **1×10⁻³**，训练 **100** 个 epoch；

- 在**每一个** mini-batch 更新之前、当前参数 $w_t$ 与**当前 batch** 上，沿 **+梯度方向** 做距离扫描（不写入参数更新）；

- 距离倍数取 $\alpha \in \{0.5,\,1,\,2,\,4\}$，目标弧长为 $s=\alpha\eta$（$\eta=10^{-3}$ 为训练学习率），实际位移长度取 $\|\delta\|=\min(s,\,0.4\|g_t\|)$，方向为 $d_t=g_t/\|g_t\|$，扰动点 $w_t'=w_t+\|\delta\|\,d_t$。

下面给出**梯度预测性**与**梯度随距离变化最大差值**的具体定义。记 $g_t=\nabla L(w_t)$。对每个 $\alpha$ 在 $w_t'$ 上再算梯度，定义

$$
D_t(\alpha)=\big\|g(w_t')-g_t\big\|_2 .
$$

先给出梯度预测性的定义。在同一步 $t$、**同一** $w_t$ 上，$\{D_t(\alpha)\}$ 随距离的变化反映梯度场在局部是否稳定；绘图取

$$
D_t^{\min}=\min_\alpha D_t(\alpha), \qquad D_t^{\max}=\max_\alpha D_t(\alpha),
$$

以填充区域展示**对距离**的波动范围。$D_t^{\max}$ 越小、$D_t^{\max}-D_t^{\min}$ 越窄，表示沿梯度方向挪动时梯度向量变化更温和、一阶近似更可信。

再给出梯度随距离变化最大差值的定义。在同一步对距离中取最大，也即

$$
\Delta_t=\max_\alpha D_t(\alpha)=D_t^{\max}.
$$

$\Delta_t$ 是 effective $\beta$-smoothness 在训练轨迹上的逐步估计：$\Delta_t$ 越大，局部曲面越尖锐；BN 若使优化 landscape 更平滑，期望 $\Delta_t$ 整体更低，且随训练推进更稳定。

训练实现中完整记录向量 $\{D_t(\alpha)\}$ 与每步的 $L(w_t)$、$L(w_t')$ ；本组**不**使用 2.4 中八组多学习率包络来代理距离扫描。可视化见 2.5.2。

本实验的目的是，在选定学习率下、于真实的每一步 $w_t$ 上比较有 BN 与无 BN 的梯度随距离变化，从梯度侧验证 BN 是否使 landscape 更利于一阶优化；与 2.4 的损失包络相互补充。

#### 2.5.2. 实验结果

下表为梯度实验（固定 $\eta=10^{-3}$，100 epoch）分类性能汇总；训练结束后在测试集上评估一次。

| 实验 | 最优验证集准确率 | 最优 epoch | 测试集准确率 |
|------|------------------|------------|--------------|
| VGG-A（无 BN） | 0.8534 | 81 | 0.8451 |
| VGG-A + BN | 0.8916 | 100 | 0.8885 |

由表可知，在相同 $\eta=10^{-3}$、100 epoch 的梯度扫描训练设定下，加 BN 后验证/测试准确率均高于无 BN（0.8885 vs. 0.8451），与 2.3、2.4 结论一致。

需特别说明的是，有两组 loss_landscape 中的实验与 grad 的两组实验虽学习率、epoch、优化器与数据划分相同，却是两次从头开始的独立训练，并非同一条参数轨迹上的两种记录。二者差别主要来源于每个 step 的训练动力学不同：loss_landscape 在常规反传后只做一次轻量扰动探测，而 grad 在 `optimizer.step()` 前要对当前 $w_t$ 沿梯度方向做四档距离扫描，每档都额外前向、反传并在参数上临时加减 $\delta$ 再恢复。尽管恢复后会在 $w_t$ 上重新反传再更新，但中间多出的计算会改变随机数消耗顺序（影响 shuffle、随机裁剪等）、带来浮点舍入差异，并使各 epoch 的 batch 顺序逐步分叉，故 100 个 epoch 后权重与最优 checkpoint 出现了不一致。因此 2.5 表仅作该设定下的性能参考，梯度 landscape 的可比性应看同一次 grad 训练内的 grad_sweep 曲线，而非与 2.4 同 lr 条目逐数对齐。

图为两种模型在训练全程中、同一步、对距离 $\alpha$ 取 min/max 的 $D_t(\alpha)$ 包络（指标 2）；出于绘图美观性考虑，实际绘图中每 44 个 mini-batch 记录一个点，也即每个 epoch 记录 2 个点，并对曲线做轻度滑动平均。输出中每 1 个 mini-batch 均完成四次距离扫描，逐步的 $D_t(\alpha)$ 在训练日志中均有保存。

![2.5.2 梯度预测性（距离包络）](pic/2_4_2_grad_predictability_landscape.png)

由图可知，绿色（无 BN）填充带宽且整体偏高，约在 step 1000 后长期落在 0.8–1.5 一带，说明在 $\alpha\in\{0.5,1,2,4\}\times\eta$ 范围内梯度差随距离起伏大、局部梯度场不稳定；红色（+BN）前期亦有一次抬升（约 step 800 附近接近 1.6），但自 step 4000 起明显下移并收窄，后期多落在 0.2–0.4，带隙远窄于无 BN。二者在训练前半段仍有重叠，中后期则清晰分离，表明 BN 使**同一 $w_t$ 上**沿梯度方向的梯度向量变化更集中，梯度预测性更好，与 2.4 从损失侧看到的平滑趋势一致。

下图为两种模型的 $\Delta_t=\max_\alpha D_t(\alpha)$ 随训练步变化（指标 3）；出于绘图美观性考虑，抽样方式与上一图相同（每 44 个 mini-batch 一点、每 epoch 2 点，含轻度滑动平均），逐步 $\Delta_t$ 在训练日志中均有保存。

![2.5.2 梯度随距离的最大差值](pic/2_4_2_grad_diff_landscape.png)

由图可知，绿色（无 BN）在 step 1000–1200 附近达到全程峰值（约 1.7），此后虽缓慢回落，但至 step 8000 仍多在 0.8 以上，末期约 0.4；红色（+BN）峰值略早、略低（约 step 800、1.6），自 step 1500 起持续低于绿色，step 4000 后稳定在 0.6 以下，末期可至约 0.2。说明 BN 降低了沿梯度方向允许距离内的**最大梯度跳变**，梯度场更接近 Lipschitz、固定步长更新更安全；早期二者同处高 loss 区时差距不大，优势主要体现在训练中后期，与上一张包络图及 2.3、2.4 的结论相互印证。

### 2.6. 总结

在 **2.3** 的 compare 实验中，于 AdamW、余弦学习率、200 epoch 的常规训练设定下，VGG-A + BN 的验证/测试准确率由 0.8693 升至 **0.9006**，且更早达到最优 epoch；训练曲线上验证集上升更快，无 BN 模型后期出现验证 loss 抬升而训练 loss 仍降的过拟合迹象，加 BN 后 train/val 更贴合。**2.3.3** 从机制上补充了上述性能差异的来源：无 BN 时 S3 正值比例约 **0.10**、激活近零尖峰对应 ReLU 链式截断与层间尺度漂移；+BN 通过逐层 $(I-\mu)/\sigma$ 重参数化将 S3 正值比例提升至约 **0.47**，S5 全局 std 由 **2.25** 降至 **1.13**，并在高置信错例上呈现更聚焦的 S3 响应。这说明 BN 的收益既在**分类指标**，也在**中间表征可利用率**与**梯度传递条件**的改善，为 2.4–2.5 的 landscape 分析提供了性能与特征双面对照。

**2.4** 的损失曲面实验从**损失标量**侧刻画优化行为：在四种固定学习率、100 epoch 的八组独立训练中，+BN 在各 lr 下测试准确率整体高于无 BN，且同一步训练进度上、不同 lr 引起的 train loss 上下界包络更窄、更低，尤其在训练中后期差距明显。这表明 BN 使目标函数对**步长（学习率）** 不那么敏感，局部 loss landscape 更平滑，与 compare 实验中观察到的更稳收敛相互呼应。

**2.5** 的梯度实验进一步在**同一参数点、沿梯度方向的多距离扫描**上验证上述判断：固定扰动幅度 $\eta=10^{-3}$ 的两条训练中，梯度预测性曲线显示无 BN 模型中后期 $D_t^{\max}$ 长期偏高且包络带宽大，+BN 模型的包络区间则在约 step 4000 后明显下移并收窄；梯度随距离变化最大差值曲线亦显示 +BN 模型的$\Delta_t$自训练中前期后即持续低于无 BN 模型，末期最大梯度跳变更小。二者共同说明了， BN 不仅压低损失波动，也使**梯度向量在局部邻域内变化更温和**，一阶近似的可信度更高。

综上，本任务第二部分从**性能—损失—梯度**三条线形成一致证据：BN 在 CIFAR-10 与 VGG-A 上提升精度与泛化，使固定步长下的 loss 包络更紧，并在真实的每步 $w_t$ 上降低 $D_t(\alpha)$ 与 $\Delta_t$，从而支持**BN 通过重参数化使优化 landscape 更平滑、更有利于基于梯度的一阶优化**这一解释；三条实验相互补充，其中基线对比实验回答**BN从验证集性能上来看是否更好**，损失曲面回答**目标值对步长是否在损失数值意义上更稳**，梯度实验回答**梯度场在局部是否更平滑可预测**。2.3 节训练集与验证集曲线均表明 +BN 组更早达到较优指标，与表中 +BN 于 epoch 170、无 BN 于 epoch 175 达到验证峰值一致，从实验现象上印证了 BN 对训练收敛速度的加速作用。


## 3. 代码链接与模型权重链接

**代码仓库**：https://github.com/Jacky23307110248/CS30064-NeuralNetwork-DeepLearning/tree/main/PJ2

**数据集与模型权重（Google Drive）**：https://drive.google.com/drive/folders/13fio3sFs1qT7FBt6_mwabVRHcdk42U1d?usp=sharing

网盘中 **data** 目录为 CIFAR-10 数据，对应本地 `PJ2/data`；**outputs** 目录为各实验训练输出（含 `best.pt` 等 checkpoint 及日志），对应本地 `PJ2/outputs`。